# Live tables

A random-walk ticker appends trades in the background; the session runs a cycle every 100 ms, and each displayed table re-renders when a cycle changes it.

In [1]:
import random
import threading
import time

import pyarrow as pa
import ursid as us

s = us.local()
ticks = s.input_table(
    {
        "sym": us.string(),
        "price": us.float64(),
        "size": us.int64(),
        "ts": us.timestamp(),
    }
)
stats = ticks.agg_by(
    "sym",
    n="count(*)",
    avg="mean(price)",
    vol="std(price)",
    low="min(price)",
    high="max(price)",
)
latest = ticks.where("size > 100").update("notional = price * size").last_by("sym")
s.start("100ms")

In [2]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOG", "META", "TSLA", "AVGO", "JPM", "V"]
prices = dict.fromkeys(SYMBOLS, 100.0)
producing = threading.Event()


def produce() -> None:
    """Appends 200 trades every 50 ms until `producing` is cleared."""
    while producing.is_set():
        syms = random.choices(SYMBOLS, k=200)
        price = []
        for sym in syms:
            prices[sym] = max(0.01, prices[sym] + random.gauss(0, 0.05))
            price.append(round(prices[sym], 2))
        now = time.time_ns()
        ticks.append(
            pa.table(
                {
                    "sym": syms,
                    "price": price,
                    "size": [random.randint(1, 500) for _ in syms],
                    "ts": pa.array([now] * len(syms), pa.timestamp("ns", tz="UTC")),
                }
            )
        )
        time.sleep(0.05)


producing.set()
threading.Thread(target=produce, daemon=True).start()

In [3]:
stats

sym utf8,n int64,avg float64,vol float64,low float64,high float64
AMZN,25030,98.61360007990423,2.2084898451413832,93.51,103.67
TSLA,25291,99.44689415207007,3.3674461010976966,91.76,105.74
GOOG,24914,99.28717387814063,3.758348518617568,91.51,105.5
AVGO,24920,100.97642215088284,2.8364881856435233,95.35,106.78
NVDA,25084,101.87748166161691,1.2391789244624263,99.26,105.88
META,25187,102.38444475324575,4.065070734419087,94.32,109.58
JPM,25169,98.63246612896822,2.346678046836848,93.22,107.04
AAPL,24947,95.90350342726582,1.9661812093706192,90.58,100.71
MSFT,25048,100.34239140849546,2.493080387097467,94.3,104.44
V,25010,106.778431827269,3.6623113465497306,99.74,113.28


In [4]:
latest

sym utf8,price float64,size int64,ts timestamp,notional float64
AVGO,95.43,448,2026-11-11 04:26:29.540879,42752.64
MSFT,102.55,121,2026-11-11 04:26:29.540879,12408.55
META,96.86,209,2026-11-11 04:26:29.540879,20243.74
JPM,106.15,443,2026-11-11 04:26:29.540879,47024.450000000004
NVDA,101.92,197,2026-11-11 04:26:29.540879,20078.24
GOOG,91.97,478,2026-11-11 04:26:29.540879,43961.659999999996
AAPL,93.58,442,2026-11-11 04:26:29.540879,41362.36
V,111.26,426,2026-11-11 04:26:29.540879,47396.76
TSLA,94.01,228,2026-11-11 04:26:29.540879,21434.280000000002
AMZN,93.64,132,2026-11-11 04:26:29.540879,12360.48


In [ ]:
producing.clear()  # stop the ticker; `s.stop()` stops the cycles